In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error

from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.multioutput import MultiOutputRegressor
from scipy.stats import randint

import os
import joblib

In [2]:
# 1- xgboost
# 2- adaboost
# 3- gradient boosting
# 4- catboost
# 5- lightboost
# 6- random forest
# 7- stacking 01
# 8- stacking 02

In [3]:
df = pd.read_csv("../data/preprocessed_dataset.csv")

df.head(2)

,FA (kg/m3),GGBFS (kg/m3),Coarse aggregate (kg/m3),Fine aggregate (kg/m3),Na2SiO3,NaOH,Sio2 (Dry),Concentration (M) NaOH,Superplasticizer,Total water,Initial curing time (day),Initial curing temp (C),Initial curing rest time (day),Final curing temp (C),Water_to_Binder,Water_to_Activator,SP_ratio,total_material_mass,Total_CO2_kg,Compressive_strength_MPa
0,0.000000,0.069565,-0.127973,-0.495765,0.700467,0.741549,0.756779,0.0,0.095238,0.651360,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.107509,0.885142,512.210033,46.57
1,0.140762,0.162319,-0.465568,0.198785,0.900600,0.950927,0.972882,0.0,0.142857,0.849623,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.159822,0.948367,534.887466,46.01


In [4]:
labels = df[['Total_CO2_kg', 'Compressive_strength_MPa']]
df.drop(columns=['Total_CO2_kg', 'Compressive_strength_MPa'], inplace=True)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(df, labels, random_state=42, test_size=0.20)

# adaboost

In [6]:
base_ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    random_state=42
)

model = MultiOutputRegressor(base_ada)

param_dist = {
    "estimator__n_estimators": randint(50, 500),
    "estimator__learning_rate": np.linspace(0.01, 1.0, 30),

    "estimator__estimator__max_depth": randint(1, 10),
    "estimator__estimator__min_samples_split": randint(2, 20),
    "estimator__estimator__min_samples_leaf": randint(1, 20),
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=100,
    scoring="r2",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=4,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV R2:", random_search.best_score_)

Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Best params: {'estimator__estimator__max_depth': 8, 'estimator__estimator__min_samples_leaf': 4, 'estimator__estimator__min_samples_split': 5, 'estimator__learning_rate': 0.31724137931034485, 'estimator__n_estimators': 406}
Best CV R2: 0.9506337858821878


In [7]:
best = random_search.best_params_


model = MultiOutputRegressor(base_ada)

grid = {
    "estimator__n_estimators": sorted(set([
        max(10, best["estimator__n_estimators"] - 40),
        best["estimator__n_estimators"],
        best["estimator__n_estimators"] + 40,
    ])),
    "estimator__learning_rate": sorted(set([
        max(0.001, best["estimator__learning_rate"] * 0.85),
        best["estimator__learning_rate"],
        min(2.0, best["estimator__learning_rate"] * 1.15),
    ])),
    "estimator__estimator__max_depth": sorted(set([
        max(1, best["estimator__estimator__max_depth"] - 1),
        best["estimator__estimator__max_depth"],
        best["estimator__estimator__max_depth"] + 1,
    ])),

    "estimator__estimator__min_samples_split": [best["estimator__estimator__min_samples_split"]],
    "estimator__estimator__min_samples_leaf":  [best["estimator__estimator__min_samples_leaf"]],
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=grid,
    scoring="r2",
    cv=cv,
    verbose=2,
    n_jobs=4,
    error_score="raise"
)

grid_search.fit(X_train, y_train)

print("Best params (Grid):", grid_search.best_params_)
print("Best R2 (Grid):", grid_search.best_score_)


Fitting 10 folds for each of 27 candidates, totalling 270 fits
Best params (Grid): {'estimator__estimator__max_depth': 9, 'estimator__estimator__min_samples_leaf': 4, 'estimator__estimator__min_samples_split': 5, 'estimator__learning_rate': 0.31724137931034485, 'estimator__n_estimators': 366}
Best R2 (Grid): 0.9509406712319979
